In [1]:
import pandas as pd

In [2]:
df_metricas_baselines = pd.read_csv("metricas_baselines.csv")
df_metricas_sarima = pd.read_csv("metricas_sarima.csv")

df_metricas = pd.concat(
    [df_metricas_baselines, df_metricas_sarima],
    ignore_index=True
)

display(
    df_metricas.sort_values(["series", "mae"])
)

,series,modelo,mae,rmse,mase
13,FOODS,sarima,186.956692,232.691803,0.546517
6,FOODS,naive_sazonal,343.857143,420.116140,1.005172
4,FOODS,media,526.928571,671.830058,1.540331
5,FOODS,naive,664.642857,722.859451,1.942901
7,FOODS,drift,664.965461,723.256672,1.943844
14,HOBBIES,sarima,76.445457,95.784893,0.837233
10,HOBBIES,naive_sazonal,102.250000,146.652773,1.119845
8,HOBBIES,media,110.994790,141.305168,1.215619
9,HOBBIES,naive,169.035714,204.727606,1.851285
11,HOBBIES,drift,170.304785,205.901357,1.865184


In [3]:
df_metricas.to_csv("metricas.csv", index=False)

In [4]:
df_metricas_ordenadas = df_metricas.sort_values(
    ["series", "mae"]
)

display(df_metricas_ordenadas)

,series,modelo,mae,rmse,mase
13,FOODS,sarima,186.956692,232.691803,0.546517
6,FOODS,naive_sazonal,343.857143,420.116140,1.005172
4,FOODS,media,526.928571,671.830058,1.540331
5,FOODS,naive,664.642857,722.859451,1.942901
7,FOODS,drift,664.965461,723.256672,1.943844
14,HOBBIES,sarima,76.445457,95.784893,0.837233
10,HOBBIES,naive_sazonal,102.250000,146.652773,1.119845
8,HOBBIES,media,110.994790,141.305168,1.215619
9,HOBBIES,naive,169.035714,204.727606,1.851285
11,HOBBIES,drift,170.304785,205.901357,1.865184


In [5]:
melhores_modelos = (
    df_metricas
    .loc[df_metricas.groupby("series")["mae"].idxmin()]
    .sort_values("series")
)

display(melhores_modelos)

,series,modelo,mae,rmse,mase
13,FOODS,sarima,186.956692,232.691803,0.546517
14,HOBBIES,sarima,76.445457,95.784893,0.837233
12,store_total,sarima,276.295978,363.297381,0.635139


In [6]:
display(
    df_metricas.sort_values(
        ["series", "mase"]
    )
)

,series,modelo,mae,rmse,mase
13,FOODS,sarima,186.956692,232.691803,0.546517
6,FOODS,naive_sazonal,343.857143,420.116140,1.005172
4,FOODS,media,526.928571,671.830058,1.540331
5,FOODS,naive,664.642857,722.859451,1.942901
7,FOODS,drift,664.965461,723.256672,1.943844
14,HOBBIES,sarima,76.445457,95.784893,0.837233
10,HOBBIES,naive_sazonal,102.250000,146.652773,1.119845
8,HOBBIES,media,110.994790,141.305168,1.215619
9,HOBBIES,naive,169.035714,204.727606,1.851285
11,HOBBIES,drift,170.304785,205.901357,1.865184


In [7]:
comparacao = df_metricas.pivot(
    index="series",
    columns="modelo",
    values=["mae", "rmse", "mase"]
)

display(comparacao)

mae                                                    \
modelo            drift       media       naive naive_sazonal      sarima   
series                                                                      
FOODS        664.965461  526.928571  664.642857    343.857143  186.956692   
HOBBIES      170.304785  110.994790  169.035714    102.250000   76.445457   
store_total  863.278662  813.885203  862.750000    459.178571  276.295978   

                   rmse                                                     \
modelo            drift        media       naive naive_sazonal      sarima   
series                                                                       
FOODS        723.256672   671.830058  722.859451    420.116140  232.691803   
HOBBIES      205.901357   141.305168  204.727606    146.652773   95.784893   
store_total  966.756301  1109.594703  966.632280    655.467091  363.297381   

                 mase                                              
modelo          drift     media     naive naive_sazonal    sarima  
series                                                             
FOODS        1.943844  1.540331  1.942901      1.005172  0.546517  
HOBBIES      1.865184  1.215619  1.851285      1.119845  0.837233  
store_total  1.984473  1.870929  1.983258      1.055543  0.635139

* FOODS: o SARIMA teve um erro bem menor que o segundo melhor modelo, o naive sazonal.

* HOBBIES: o SARIMA também foi o melhor.

* Pra stor_total: o SARIMA também venceu com uma diferença consideráel.



In [8]:
comparacao_melhora = []

for serie in df_metricas["series"].unique():
    dados = df_metricas[df_metricas["series"] == serie]

    mae_sarima = dados.loc[
        dados["modelo"] == "sarima", "mae"
    ].iloc[0]

    mae_naive_sazonal = dados.loc[
        dados["modelo"] == "naive_sazonal", "mae"
    ].iloc[0]

    melhora = (
        (mae_naive_sazonal - mae_sarima)
        / mae_naive_sazonal
    ) * 100

    comparacao_melhora.append({
        "series": serie,
        "mae_sarima": mae_sarima,
        "mae_naive_sazonal": mae_naive_sazonal,
        "melhora_percentual": melhora
    })

df_melhora = pd.DataFrame(comparacao_melhora)

display(df_melhora)

,series,mae_sarima,mae_naive_sazonal,melhora_percentual
0,store_total,276.295978,459.178571,39.828207
1,FOODS,186.956692,343.857143,45.629545
2,HOBBIES,76.445457,102.250000,25.236717


## Conclusão

O modelo SARIMA apresentou desempenho superior aos quatro baselines considerados — Média, Naive, Naive sazonal e Drift — nas três séries analisadas.

Na série `store_total`, o SARIMA obteve MAE de **276,30**, contra **459,18** do melhor baseline, o Naive sazonal, representando uma redução de aproximadamente **39,8%** no erro. Para `FOODS`, o ganho foi ainda maior: o MAE passou de **343,86** no Naive sazonal para **186,96** no SARIMA, uma redução de aproximadamente **45,6%**. Já em `HOBBIES`, o MAE foi reduzido de **102,25** para **76,45**, correspondendo a uma melhora de aproximadamente **25,2%**.

O mesmo comportamento é observado no RMSE, indicando que o SARIMA também apresentou menor impacto de erros de maior magnitude. Além disso, os valores de MASE foram inferiores a 1 para todas as séries — **0,635 em `store_total`, 0,547 em `FOODS` e 0,837 em `HOBBIES`** —, indicando desempenho superior ao benchmark de referência utilizado no cálculo dessa métrica.

Entre os baselines, o **Naive sazonal foi o melhor em todas as séries**, evidenciando a importância da sazonalidade semanal nas vendas diárias. Os modelos SARIMA selecionados incorporaram essa estrutura por meio do período sazonal \(s=7\).

Por fim, o melhor desempenho relativo do SARIMA ocorreu na série `FOODS`, enquanto `HOBBIES` apresentou o maior MASE entre as três séries. Ainda assim, o SARIMA superou os baselines em todas as métricas e séries avaliadas, indicando que a combinação entre dependência temporal e sazonalidade semanal foi adequada para a tarefa de previsão.